In [16]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from dotenv import load_dotenv

In [5]:
load_dotenv()

chat_mistral = HuggingFaceEndpoint(
    repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
    task="text-generation",
    temperature=1.2,
)

model = ChatHuggingFace(llm=chat_mistral)

In [6]:
documents = [

    {
        "id": 1,
        "text": """FastAPI is a modern Python web framework used to build APIs. 
        It is known for high performance and async support. 
        FastAPI also provides automatic API documentation using Swagger UI. 
        Many developers use it for building scalable backend systems. 
        Python itself is a popular programming language created by Guido van Rossum.""",
        "metadata": {"topic": "fastapi"}
    },

    {
        "id": 2,
        "text": """MongoDB is a NoSQL database that stores data in JSON-like documents. 
        It is highly scalable and flexible. 
        MongoDB is often used in MERN stack applications. 
        The capital of France is Paris. 
        Databases are important for storing application data.""",
        "metadata": {"topic": "mongodb"}
    },

    {
        "id": 3,
        "text": """Docker is a containerization platform that allows developers to package applications. 
        It ensures consistency across environments. 
        Docker uses images and containers. 
        Cricket is a very popular sport in India. 
        Containers help in deployment and scaling.""",
        "metadata": {"topic": "docker"}
    },

    {
        "id": 4,
        "text": """Embeddings convert text into numerical vectors. 
        These vectors are used in machine learning and similarity search. 
        Vector databases store embeddings for efficient retrieval. 
        Mumbai is a financial hub of India. 
        AI applications rely heavily on embeddings.""",
        "metadata": {"topic": "embeddings"}
    }

]

In [7]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cpu"}
)

In [8]:
vector_store = FAISS.from_documents(
    documents = [Document(page_content=doc["text"], metadata=doc["metadata"]) for doc in documents],
    embedding = embedding
)

In [9]:
docs_dict = vector_store.docstore._dict

for doc_id, doc in docs_dict.items():
    print(doc_id, doc.page_content)

126a39ce-7454-46a8-a494-e6e23df0065e FastAPI is a modern Python web framework used to build APIs. 
        It is known for high performance and async support. 
        FastAPI also provides automatic API documentation using Swagger UI. 
        Many developers use it for building scalable backend systems. 
        Python itself is a popular programming language created by Guido van Rossum.
21edb555-e129-4013-ab74-7400bdcb8c9a MongoDB is a NoSQL database that stores data in JSON-like documents. 
        It is highly scalable and flexible. 
        MongoDB is often used in MERN stack applications. 
        The capital of France is Paris. 
        Databases are important for storing application data.
bfcb1526-9077-4f91-9c1e-c152dd7f52e4 Docker is a containerization platform that allows developers to package applications. 
        It ensures consistency across environments. 
        Docker uses images and containers. 
        Cricket is a very popular sport in India. 
        Containers he

In [29]:
base_retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [30]:
compressor = LLMChainExtractor.from_llm(model)

In [31]:
compressor_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [32]:
query = "What is fastapi?"

In [33]:
compressed_results = compressor_retriever.invoke(query)

In [34]:
for i,result in enumerate(compressed_results):
    print(f"Result {i+1}: {result.page_content}")

Result 1: FastAPI is a modern Python web framework used to build APIs. 
        It is known for high performance and async support. 
        FastAPI also provides automatic API documentation using Swagger UI.
Result 2: >>
AI applications rely heavily on embeddings.
>>
Result 3: No relavent context was found showcasing what fastapi is.
Result 4: >>> MongoDB is a NoSQL database that stores data in JSON-like documents. 
      It is highly scalable and flexible. 
      MongoDB is often used in MERN stack applications. 
      Databases are important for storing application data. 
      FastAPI is a modern, fast (high-performance), web framework for building APIs with Python 3.7+ based on standard Python type hints).


 Dosent exsist within the Artifical int between this contront note tho.
